# SMMILe: Ovarian Cancer 5-Subtype WSI Classification

This notebook runs the **original SMMILe code** (Gao et al., Nature Cancer 2025) for ovarian cancer subtype classification on UBC-OCEAN (513 WSIs, 5 subtypes: HGSC, EC, CC, LGSC, MC).

**Encoder:** Conch (pathology foundation model, 512-dim embeddings)

**Requirements:**
- Google Colab with GPU (A100 recommended for speed, T4/V100 also work)
- Pre-extracted Conch embeddings and superpixel files on Google Drive
  - Download from HuggingFace: [zeyugao/SMMILe_Datasets](https://huggingface.co/datasets/zeyugao/SMMILe_Datasets)

**Pipeline:**
1. Setup & install dependencies
2. Configure data paths
3. Generate 5-fold cross-validation splits
4. Train (5 folds, ~50 epochs each)
5. Evaluate & report metrics

## 0. GPU Check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
gpu_name = result.stdout.strip()
print('GPU:', gpu_name)
if 'A100' in gpu_name:
    print('A100 detected - optimal performance.')
elif gpu_name:
    print('Non-A100 GPU detected. Training will work but may be slower.')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU')

## 1. Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted.')

In [ ]:
import os

REPO_DIR = '/content/SQ-MIL'
REPO_URL = 'https://github.com/z-pan/SQ-MIL.git'
BRANCH = 'claude/train-stage1-fold0-AuGpr'

if os.path.isdir(REPO_DIR):
    print('Repo already cloned - pulling latest...')
    !git -C {REPO_DIR} fetch origin {BRANCH} && git -C {REPO_DIR} reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(os.path.join(REPO_DIR, 'smmile'))
print('Working directory:', os.getcwd())

## 2. Install Dependencies

In [ ]:
!pip install -q tensorboardX pyyaml scikit-learn scikit-image scipy pandas h5py tqdm matplotlib opencv-python-headless
print('Dependencies installed.')

## 3. Configuration

**Edit the paths below** to point to your data on Google Drive.

Expected data layout:
- **Embeddings dir**: Contains `.npy` files like `{slide_id}_0_512.npy` (one per WSI)
- **Superpixels dir**: Contains `.npy` files like `{slide_id}_0_512.npy` (one per WSI)

These are the pre-extracted files from [zeyugao/SMMILe_Datasets](https://huggingface.co/datasets/zeyugao/SMMILe_Datasets) on HuggingFace.

In [ ]:
# ============================================================
# USER CONFIGURATION - Edit these paths
# ============================================================

# Path to Conch embedding .npy files on Google Drive
EMBED_DIR = '/content/drive/MyDrive/SQ-MIL/ovarian_conch/conch'

# Path to superpixel .npy files on Google Drive
SP_DIR = '/content/drive/MyDrive/SQ-MIL/ovarian_conch/sp_conch_n16_c50_512'

# Where to save results (checkpoints, logs, eval output)
# Using Drive so results persist across sessions
RESULTS_DIR = '/content/drive/MyDrive/SQ-MIL/results_conch_ov'

# ============================================================
# Training settings (match the published config)
# ============================================================
N_FOLDS = 5
MAX_EPOCHS = 50
SEED = 1

# Config file (relative to smmile/ directory)
CONFIG = 'configs_ov/config_ovarian_smmile_r1_conch.yaml'

# ============================================================
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'EMBED_DIR  : {EMBED_DIR}')
print(f'SP_DIR     : {SP_DIR}')
print(f'RESULTS_DIR: {RESULTS_DIR}')
print(f'N_FOLDS    : {N_FOLDS}')
print(f'MAX_EPOCHS : {MAX_EPOCHS}')

## 4. Pre-flight Validation

In [ ]:
import glob
import pandas as pd
import torch

errors = []

# Check embeddings
emb_files = glob.glob(os.path.join(EMBED_DIR, '*_0_512.npy'))
if len(emb_files) == 0:
    # Try without pattern
    emb_files = glob.glob(os.path.join(EMBED_DIR, '*.npy'))
if len(emb_files) == 0:
    errors.append(f'No .npy files found in EMBED_DIR: {EMBED_DIR}')
else:
    print(f'Embeddings: {len(emb_files)} files found')

# Check superpixels
sp_files = glob.glob(os.path.join(SP_DIR, '*_0_512.npy'))
if len(sp_files) == 0:
    sp_files = glob.glob(os.path.join(SP_DIR, '*.npy'))
if len(sp_files) == 0:
    errors.append(f'No .npy files found in SP_DIR: {SP_DIR}')
else:
    print(f'Superpixels: {len(sp_files)} files found')

# Check dataset CSV
csv_path = 'dataset_csv/ovarian_subtyping_npy.csv'
if os.path.isfile(csv_path):
    df = pd.read_csv(csv_path)
    print(f'Dataset CSV: {len(df)} slides')
    print(f'  Labels: {df["label"].value_counts().to_dict()}')
else:
    errors.append(f'Dataset CSV not found: {csv_path}')

# Check GPU
if torch.cuda.is_available():
    print(f'CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)')
else:
    errors.append('No CUDA GPU detected.')

if errors:
    print('\nPre-flight FAILED:')
    for e in errors:
        print(f'  - {e}')
else:
    print('\nAll checks passed.')

## 5. Generate Cross-Validation Splits

Creates 5-fold stratified splits (80% train+val / 20% test; 90/10 train/val within train+val).

Splits are saved to `smmile/splits/ovarian_subtype_100/splits_{fold}.csv`.

Note: The directory name `_100` comes from `label_frac=1.0` (100% of labels used), which is what `main.py` expects by default.

In [ ]:
import numpy as np

task = 'ovarian_subtype'
csv_path = 'dataset_csv/ovarian_subtyping_npy.csv'
k_folds = N_FOLDS

# main.py computes split_dir as: splits/{task}_{int(label_frac*100)}
# With label_frac=1.0, this is splits/ovarian_subtype_100
LABEL_FRAC = 1.0
split_dir = 'splits/' + str(task) + '_{}'.format(int(LABEL_FRAC * 100))
os.makedirs(split_dir, exist_ok=True)

# Check if splits already exist
existing_splits = glob.glob(os.path.join(split_dir, 'splits_*.csv'))
if len(existing_splits) >= k_folds:
    print(f'Splits already exist in {split_dir} ({len(existing_splits)} files). Skipping generation.')
    for sf in sorted(existing_splits):
        d = pd.read_csv(sf)
        print(f'  {os.path.basename(sf)}: train={d["train"].dropna().shape[0]}, val={d["val"].dropna().shape[0]}, test={d["test"].dropna().shape[0]}')
else:
    print(f'Generating {k_folds}-fold splits into {split_dir} ...')
    dataset = pd.read_csv(csv_path)

    # For UBC-OCEAN, case_id == slide_id (one slide per case)
    if 'case_id' not in dataset.columns:
        dataset['case_id'] = dataset['slide_id']

    label_set = np.unique(dataset['label'])
    np.random.seed(SEED)

    case_id_to_fold = {}
    for label in label_set:
        dataset_sub = dataset[dataset['label'] == label]
        case_ids = dataset_sub['case_id'].unique()
        np.random.shuffle(case_ids)
        case_id_to_fold.update({case_id: i % k_folds for i, case_id in enumerate(case_ids)})

    dataset['k_fold'] = dataset['case_id'].map(case_id_to_fold)

    for i in range(k_folds):
        train_val_set = dataset[dataset['k_fold'] != i].reset_index(drop=True)
        test_set = dataset[dataset['k_fold'] == i].reset_index(drop=True)

        train_set = []
        val_set = []
        for label in label_set:
            train_val_class = train_val_set[train_val_set['label'] == label].copy()
            case_ids = train_val_class['case_id'].unique()
            np.random.shuffle(case_ids)
            split_idx = int(len(case_ids) * 0.9)
            train_case_ids = case_ids[:split_idx]
            val_case_ids = case_ids[split_idx:]
            train_set.append(train_val_class[train_val_class['case_id'].isin(train_case_ids)])
            val_set.append(train_val_class[train_val_class['case_id'].isin(val_case_ids)])

        train_set = pd.concat(train_set, ignore_index=True)
        val_set = pd.concat(val_set, ignore_index=True)

        df_split = pd.DataFrame({
            'train': pd.Series(train_set['slide_id'].values),
            'val': pd.Series(val_set['slide_id'].values),
            'test': pd.Series(test_set['slide_id'].values)
        })

        split_path = os.path.join(split_dir, 'splits_{}.csv'.format(i))
        df_split.to_csv(split_path, index=False)
        print(f'  Fold {i}: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}')

    print(f'Splits saved to: {split_dir}')

## 6. Training

Runs the original SMMILe `main.py` with the published ovarian Conch config.

**Published config summary:**
- Model: SMMILe `small` (NIC=128, attn=64)
- Loss: BCE on probabilities
- No Instance Dropout, no Instance Sampling, no refinement, no MRF
- 50 epochs, lr=2e-5, weight_decay=1e-5, Adam
- Early stopping (patience=20, min epoch=50)
- Weighted sampling for class imbalance

This trains all 5 folds sequentially. Each fold takes ~20-40 min on A100.

In [ ]:
import subprocess, sys

def run(cmd, cwd=None):
    """Run a command, stream output, raise on failure."""
    print('$', ' '.join(str(c) for c in cmd))
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, cwd=cwd
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}')

print('run() helper defined.')

In [ ]:
# Train all folds
SMMILE_DIR = os.path.join(REPO_DIR, 'smmile')
SPLIT_DIR_NAME = 'ovarian_subtype_{}'.format(int(LABEL_FRAC * 100))

run([
    sys.executable, 'main.py',
    '--config', CONFIG,
    '--data_root_dir', EMBED_DIR,
    '--data_sp_dir', SP_DIR,
    '--results_dir', RESULTS_DIR,
    '--split_dir', SPLIT_DIR_NAME,
    '--max_epochs', str(MAX_EPOCHS),
    '--seed', str(SEED),
    '--k', str(N_FOLDS),
    '--k_start', '0',
    '--k_end', str(N_FOLDS),
], cwd=SMMILE_DIR)

print('\nTraining complete for all folds.')

### 6b. Train a Single Fold (optional)

Use this cell to train just one fold (e.g., fold 0) for a quick test.

In [ ]:
# Uncomment and run to train a single fold:

# FOLD = 0
# run([
#     sys.executable, 'main.py',
#     '--config', CONFIG,
#     '--data_root_dir', EMBED_DIR,
#     '--data_sp_dir', SP_DIR,
#     '--results_dir', RESULTS_DIR,
#     '--split_dir', SPLIT_DIR_NAME,
#     '--max_epochs', str(MAX_EPOCHS),
#     '--seed', str(SEED),
#     '--k', str(N_FOLDS),
#     '--k_start', str(FOLD),
#     '--k_end', str(FOLD + 1),
# ], cwd=SMMILE_DIR)

## 7. Evaluation

Runs the original SMMILe `eval.py` on the test split of each fold.

Outputs per-fold WSI AUC, accuracy, and instance-level AUC.

In [ ]:
# Find the experiment directory (results_dir/exp_code_s{seed}/)
import yaml

with open(os.path.join(SMMILE_DIR, CONFIG)) as f:
    cfg = yaml.safe_load(f)

exp_code = cfg.get('exp_code', 'ov_subtyping_smmile_0512_5fold')
models_exp_code = f'{exp_code}_s{SEED}'

# Verify checkpoint directory exists
ckpt_dir = os.path.join(RESULTS_DIR, models_exp_code)
if os.path.isdir(ckpt_dir):
    ckpts = glob.glob(os.path.join(ckpt_dir, 's_*_checkpoint_best.pt'))
    print(f'Checkpoint directory: {ckpt_dir}')
    print(f'Found {len(ckpts)} best checkpoints')
    for c in sorted(ckpts):
        print(f'  {os.path.basename(c)}')
else:
    print(f'WARNING: Checkpoint directory not found: {ckpt_dir}')
    print('Make sure training completed successfully.')

In [ ]:
# Create eval results directory
EVAL_DIR = os.path.join(RESULTS_DIR, 'eval')
os.makedirs(EVAL_DIR, exist_ok=True)

run([
    sys.executable, 'eval.py',
    '--data_root_dir', EMBED_DIR,
    '--data_sp_dir', SP_DIR,
    '--results_dir', RESULTS_DIR,
    '--models_exp_code', models_exp_code,
    '--save_exp_code', '_test',
    '--split', 'test',
    '--k', str(N_FOLDS),
    '--k_start', '0',
    '--k_end', str(N_FOLDS),
], cwd=SMMILE_DIR)

print('\nEvaluation complete.')

## 8. Results Summary

In [ ]:
import numpy as np

# Load training summary
summary_path = os.path.join(ckpt_dir, 'summary.csv')
if os.path.isfile(summary_path):
    df_train = pd.read_csv(summary_path)
    print('=== Training Summary ===')
    print(df_train.to_string(index=False, float_format='{:.4f}'.format))
    print()
    for col in ['test_auc', 'val_auc', 'test_acc', 'val_acc', 'test_iauc', 'val_iauc']:
        if col in df_train.columns:
            vals = df_train[col].values
            print(f'  {col:15s}: {np.mean(vals)*100:.2f} +/- {np.std(vals)*100:.2f} %')
else:
    print(f'Training summary not found at: {summary_path}')

# Load eval summary
eval_save_dir = os.path.join('./eval_results', f'EVAL_{models_exp_code}_test')
eval_summary = os.path.join(eval_save_dir, 'summary.csv')
if os.path.isfile(eval_summary):
    df_eval = pd.read_csv(eval_summary)
    print('\n=== Evaluation Summary ===')
    print(df_eval.to_string(index=False, float_format='{:.4f}'.format))
    print()
    for col in ['test_auc', 'test_acc', 'test_iauc', 'test_iacc']:
        if col in df_eval.columns:
            vals = df_eval[col].values
            print(f'  {col:15s}: {np.mean(vals)*100:.2f} +/- {np.std(vals)*100:.2f} %')
else:
    # Try alternate location
    print(f'Eval summary not found at: {eval_summary}')
    print('Check eval_results/ directory for output files.')

In [ ]:
# Visualize per-fold results
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if os.path.isfile(summary_path):
    df_plot = pd.read_csv(summary_path)
    metrics_to_plot = [c for c in ['test_auc', 'test_acc', 'test_iauc'] if c in df_plot.columns]

    if metrics_to_plot:
        fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(5*len(metrics_to_plot), 4))
        if len(metrics_to_plot) == 1:
            axes = [axes]

        for ax, metric in zip(axes, metrics_to_plot):
            vals = df_plot[metric].values * 100
            folds = df_plot['folds'].values
            ax.bar(folds, vals, color='steelblue', alpha=0.8)
            ax.axhline(np.mean(vals), color='red', linestyle='--',
                       label=f'mean={np.mean(vals):.2f}%')
            ax.set_title(metric.replace('_', ' ').title())
            ax.set_xlabel('Fold')
            ax.set_ylabel('%')
            ax.set_ylim(0, 105)
            ax.legend(fontsize=9)

        plt.suptitle('SMMILe Ovarian Conch: 5-Fold CV Results', fontsize=13, y=1.02)
        plt.tight_layout()
        plot_path = os.path.join(RESULTS_DIR, 'cv_results.png')
        plt.savefig(plot_path, bbox_inches='tight', dpi=150)
        plt.show()
        print(f'Plot saved to: {plot_path}')
    else:
        print('No metrics columns found for plotting.')
else:
    print('No training summary file to plot.')

## 9. Copy Results to Drive (Backup)

If results were saved locally (not to Drive), copy them now.

In [ ]:
# List all output files
print('=== Output Files ===')
for root, dirs, files in os.walk(RESULTS_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  {os.path.relpath(fpath, RESULTS_DIR):60s} ({size_mb:.1f} MB)')

# Also check local eval_results
local_eval = os.path.join(SMMILE_DIR, 'eval_results')
if os.path.isdir(local_eval):
    print(f'\n=== Local eval_results (copy to Drive if needed) ===')
    for root, dirs, files in os.walk(local_eval):
        for f in files:
            fpath = os.path.join(root, f)
            print(f'  {os.path.relpath(fpath, local_eval)}')

## Reference

**Paper:** Gao et al. (2025). SMMILe enables accurate spatial quantification in digital pathology using multiple-instance learning. *Nature Cancer*. DOI: 10.1038/s43018-025-01060-8

**Original Code:** https://github.com/ZeyuGaoAi/SMMILe

**Paper Results (UBC-OCEAN, Conch):** WSI AUC = 97.01%, Spatial AUC = 96.67%, Spatial F1 = 96.02%, Spatial Accuracy = 92.98%